## Kiểm tra GPU

In [1]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Kết nối Drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Extract data zip

In [4]:
import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/Doan/Dataset_Layoutlmv3/Layoutlmv3_Dataset.zip"
EXTRACT_DIR = "/content/layoutlmv3_data"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Extract done.")

Extract done.


## Kiểm tra file zip

In [5]:
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 2 * (level + 1)
    for f in files[:5]:
        print(f"{subindent}{f}")

layoutlmv3_data/
  valid/
    0_layoutlmv3_dataset_normalized.json
    images/
      pan_group_jsc_2025_p042_jpg.rf.25ce58b30bb4377896d27342a426e4de.jpg
      fpt_corp_2025_p076_jpg.rf.3082d2a7fec0fd11bffdb3f9beff87be.jpg
      bao_viet_holdings_2024_p063_jpg.rf.5ebf1ae53bef9c9389495a9c77a573d7.jpg
      vietnam_airlines_jsc_2024_p040_jpg.rf.5a0b7711ac7394fe774b87e1124dfe12.jpg
      vietnam_national_petroleum_group_2024_p076_jpg.rf.ec026fe0699961a8fda0218dc33ff689.jpg


## SET DIR

In [12]:
DATASET_DIR = "/content/layoutlmv3_data/valid/"
IMAGE_DIR = "/content/layoutlmv3_data/valid/images"
JSON_PATH = "/content/layoutlmv3_data/valid/0_layoutlmv3_dataset_normalized.json"


## Kiểm tra thư viện

In [7]:
import torch
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

Torch: 2.11.0+cu128
Transformers: 5.12.1
CUDA: True


## Kiểm tra số ảnh/trang

In [13]:
import json
import os
from collections import Counter
from PIL import Image

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of pages:", len(data))

sample = data[0]
print(sample.keys())
print("File name:", sample["file_name"])
print("Words:", len(sample["words"]))
print("Bboxes:", len(sample["bboxes"]))
print("Labels:", len(sample["labels"]))
print("Image exists:", os.path.exists(os.path.join(IMAGE_DIR, sample["file_name"])))

Number of pages: 2466
dict_keys(['image_id', 'file_name', 'width', 'height', 'words', 'bboxes', 'labels', 'bbox_scale'])
File name: bao_viet_holdings_2024_p345_jpg.rf.a30d88b69fd1694f4aa1e448d0835a10.jpg
Words: 44
Bboxes: 44
Labels: 44
Image exists: True


## Kiểm tra Missing, Error

In [14]:
missing = 0
length_error = 0
bbox_error = 0
label_counter = Counter()

for item in data:
    image_path = os.path.join(IMAGE_DIR, item["file_name"])

    if not os.path.exists(image_path):
        missing += 1

    if not (len(item["words"]) == len(item["bboxes"]) == len(item["labels"])):
        length_error += 1

    for box in item["bboxes"]:
        if len(box) != 4 or min(box) < 0 or max(box) > 1000:
            bbox_error += 1

    label_counter.update(item["labels"])

print("Missing images:", missing)
print("Length error:", length_error)
print("Bbox error:", bbox_error)
print("Label distribution:")
print(label_counter)

Missing images: 0
Length error: 0
Bbox error: 0
Label distribution:
Counter({'text': 68357, 'table': 39889, 'toc': 19656, 'table_text': 13418, 'ignore': 12777, 'figure': 12095, 'chart': 5649, 'header': 5060, 'footer': 2051, 'O': 88})


## Split dataset

In [19]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

print("Total:", len(data))
print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

Total: 2466
Train: 1972
Validation: 247
Test: 247


## Lưu Spilit

In [20]:
import json
import os

SPLIT_DIR = "/content/drive/MyDrive/Doan/Dataset_Layoutlmv3/splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

with open(f"{SPLIT_DIR}/train_data.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open(f"{SPLIT_DIR}/val_data.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

with open(f"{SPLIT_DIR}/test_data.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

print("Saved split files to:", SPLIT_DIR)

Saved split files to: /content/drive/MyDrive/Doan/Dataset_Layoutlmv3/splits


## Label Mapping

In [21]:
labels = [
    "O",
    "chart",
    "figure",
    "footer",
    "header",
    "ignore",
    "table",
    "table_text",
    "text",
    "toc"
]

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

{'O': 0, 'chart': 1, 'figure': 2, 'footer': 3, 'header': 4, 'ignore': 5, 'table': 6, 'table_text': 7, 'text': 8, 'toc': 9}


## Kiểm tra Label Mapping

In [29]:
all_labels_in_data = set()

for item in data:
    all_labels_in_data.update(item["labels"])

unknown_labels = all_labels_in_data - set(labels)
missing_labels = set(labels) - all_labels_in_data

print("Unknown labels:", unknown_labels)
print("Labels not appearing:", missing_labels)

assert len(unknown_labels) == 0, f"Dataset có label lạ: {unknown_labels}"

Unknown labels: set()
Labels not appearing: set()


## Lưu Mapping

In [22]:
LABEL_DIR = "/content/drive/MyDrive/Doan/Dataset_Layoutlmv3/label_mapping"
os.makedirs(LABEL_DIR, exist_ok=True)

with open(f"{LABEL_DIR}/label2id.json", "w", encoding="utf-8") as f:
    json.dump(label2id, f, ensure_ascii=False, indent=2)

with open(f"{LABEL_DIR}/id2label.json", "w", encoding="utf-8") as f:
    json.dump(id2label, f, ensure_ascii=False, indent=2)

print("Saved label mapping to:", LABEL_DIR)

Saved label mapping to: /content/drive/MyDrive/Doan/Dataset_Layoutlmv3/label_mapping


## Load Profesor

In [23]:
from transformers import LayoutLMv3Processor

processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

In [24]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image

class LayoutDataset(Dataset):
    def __init__(self, examples, image_dir, processor, label2id, max_length=512):
        self.examples = examples
        self.image_dir = image_dir
        self.processor = processor
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples[idx]

        image_path = os.path.join(self.image_dir, item["file_name"])
        image = Image.open(image_path).convert("RGB")

        words = item["words"]
        boxes = item["bboxes"]
        labels_item = item["labels"]

        word_labels = [self.label2id[label] for label in labels_item]

        encoding = self.processor(
            image,
            words,
            boxes=boxes,
            word_labels=word_labels,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        return encoding

In [25]:
MAX_LENGTH = 512

train_dataset = LayoutDataset(
    train_data,
    IMAGE_DIR,
    processor,
    label2id,
    max_length=MAX_LENGTH
)

val_dataset = LayoutDataset(
    val_data,
    IMAGE_DIR,
    processor,
    label2id,
    max_length=MAX_LENGTH
)

test_dataset = LayoutDataset(
    test_data,
    IMAGE_DIR,
    processor,
    label2id,
    max_length=MAX_LENGTH
)

In [26]:
sample_encoding = train_dataset[0]

for k, v in sample_encoding.items():
    print(k, v.shape)

input_ids torch.Size([512])
attention_mask torch.Size([512])
bbox torch.Size([512, 4])
labels torch.Size([512])
pixel_values torch.Size([3, 224, 224])


## Load Model

In [27]:
from transformers import LayoutLMv3ForTokenClassification

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                        | Status  | 
---------------------------+---------+-
classifier.dense.weight    | MISSING | 
classifier.out_proj.bias   | MISSING | 
classifier.out_proj.weight | MISSING | 
classifier.dense.bias      | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Tạo thư mục checkpoint trong Drive

In [28]:
import os

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Doan/Dataset_Layoutlmv3/layoutlmv3_checkpoints"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print("Checkpoint dir:", DRIVE_OUTPUT_DIR)

Checkpoint dir: /content/drive/MyDrive/Doan/Dataset_Layoutlmv3/layoutlmv3_checkpoints


In [36]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=DRIVE_OUTPUT_DIR,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=5e-5,
    num_train_epochs=10,
    weight_decay=0.01,

    logging_steps=50,

    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,

    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,
    report_to="none",

    remove_unused_columns=False
)

## Tạo Trainer

In [37]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor
)

## Training

In [38]:
train_result = trainer.train()

print(train_result)

Step,Training Loss,Validation Loss
250,4.176415,0.473471
500,3.117473,0.445299
750,2.847447,0.423347
1000,1.882460,0.366579
1250,1.479305,0.371066
1500,1.006185,0.429786
1750,0.820629,0.434348
2000,0.460363,0.442501
2250,0.261082,0.459352
2470,0.277001,0.468479


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2470, training_loss=1.9223931714108116, metrics={'train_runtime': 7722.7273, 'train_samples_per_second': 2.554, 'train_steps_per_second': 0.32, 'total_flos': 5234215095091200.0, 'train_loss': 1.9223931714108116, 'epoch': 10.0})


In [39]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)

Best checkpoint: /content/drive/MyDrive/Doan/Dataset_Layoutlmv3/layoutlmv3_checkpoints/checkpoint-1000
Best metric: 0.3665787875652313


In [40]:
test_result = trainer.evaluate(eval_dataset=test_dataset)

print("Test result:")
print(test_result)

Training Loss,Validation Loss,Step
0.277001,0.389648,2470


Test result:
{'eval_loss': 0.38964757323265076}


In [41]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels_true = predictions.label_ids

pred_ids = np.argmax(logits, axis=-1)

true_labels = []
pred_labels = []

for pred_seq, label_seq in zip(pred_ids, labels_true):
    for pred_id, label_id in zip(pred_seq, label_seq):
        if label_id != -100:
            true_labels.append(id2label[int(label_id)])
            pred_labels.append(id2label[int(pred_id)])

print("Number of evaluated tokens:", len(true_labels))

report = classification_report(
    true_labels,
    pred_labels,
    labels=labels,
    digits=4,
    zero_division=0
)

print(report)

Number of evaluated tokens: 14583
              precision    recall  f1-score   support

           O     0.0000    0.0000    0.0000         4
       chart     0.9330    0.8813    0.9064       379
      figure     0.7418    0.7229    0.7322      1061
      footer     0.8304    0.5254    0.6436       177
      header     0.8801    0.8633    0.8716       578
      ignore     0.8427    0.7184    0.7756       902
       table     0.9465    0.9676    0.9569      3364
  table_text     0.8344    0.9108    0.8709       863
        text     0.9239    0.9480    0.9358      5380
         toc     0.9940    0.9765    0.9852      1875

    accuracy                         0.9129     14583
   macro avg     0.7927    0.7514    0.7678     14583
weighted avg     0.9117    0.9129    0.9113     14583



In [42]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

accuracy = accuracy_score(true_labels, pred_labels)

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    true_labels,
    pred_labels,
    labels=labels,
    average="macro",
    zero_division=0
)

weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    true_labels,
    pred_labels,
    labels=labels,
    average="weighted",
    zero_division=0
)

micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
    true_labels,
    pred_labels,
    labels=labels,
    average="micro",
    zero_division=0
)

print("===== Overall Benchmarks =====")
print(f"Token Accuracy   : {accuracy:.4f}")
print(f"Micro Precision  : {micro_precision:.4f}")
print(f"Micro Recall     : {micro_recall:.4f}")
print(f"Micro F1         : {micro_f1:.4f}")
print(f"Macro Precision  : {macro_precision:.4f}")
print(f"Macro Recall     : {macro_recall:.4f}")
print(f"Macro F1         : {macro_f1:.4f}")
print(f"Weighted Precision: {weighted_precision:.4f}")
print(f"Weighted Recall   : {weighted_recall:.4f}")
print(f"Weighted F1       : {weighted_f1:.4f}")

===== Overall Benchmarks =====
Token Accuracy   : 0.9129
Micro Precision  : 0.9129
Micro Recall     : 0.9129
Micro F1         : 0.9129
Macro Precision  : 0.7927
Macro Recall     : 0.7514
Macro F1         : 0.7678
Weighted Precision: 0.9117
Weighted Recall   : 0.9129
Weighted F1       : 0.9113
